In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
import os

# Token'ı Colab secrets'a manuel ekleyin:
# Sol tarafta 🔑 (Key) simgesine tıklayın
# "Add new secret" > Name: HF_TOKEN > Value: YOUR_TOKEN_HERE
# hf_REDACTED
# Sonra bu kodu çalıştırın:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✅ Hugging Face token yüklendi!")
except:
    print("❌ Token bulunamadı. Lütfen Colab secrets'a ekleyin.")

✅ Hugging Face token yüklendi!


In [3]:
# SAM3 ve bağımlılıklarını kur
!pip install -q torch torchvision torchaudio
!pip install -q pycocotools timm einops psutil decord
!pip install -q pillow matplotlib
print("✅ Kurulum tamamlandı!")
gerekli = ['pycocotools', 'timm', 'einops', 'psutil','decord']

for paket in gerekli:
    try:
        __import__(paket)
        print(f"✅ {paket} yüklü")
    except ImportError:
        print(f"❌ {paket} yüklü değil - yükleniyor...")
        !pip install -q {paket}
        print(f"✅ {paket} kuruldu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 143.6 MB/s eta 0:00:00
✅ Kurulum tamamlandı!
✅ pycocotools yüklü
✅ timm yüklü
✅ einops yüklü
✅ psutil yüklü
✅ decord yüklü


In [4]:
# Önce mevcut sam3 varsa temizle
!rm -rf sam3

# GitHub'dan klonla
!git clone https://github.com/facebookresearch/sam3.git

# Klonlama başarılı mı kontrol et
import os
if os.path.exists('sam3'):
    print("✅ SAM3 klasörü klonlandı")

    # sam3 klasörüne gir ve kur
    %cd sam3
    !pip install -e .

    print("\n=== Kurulum Kontrolü ===")
    # Kontrol et
    try:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("✅ SAM3 modülleri başarıyla import edildi!")
    except Exception as e:
        print(f"❌ Import hatası: {e}")
else:
    print("❌ Klonlama başarısız. Repo adı yanlış olabilir.")
    print("Alternatif olarak SAM2'yi deneyelim:")
    !git clone https://github.com/facebookresearch/sam2.git

Cloning into 'sam3'...
remote: Enumerating objects: 578, done.
remote: Total 578 (delta 0), reused 0 (delta 0), pack-reused 578 (from 1)
Receiving objects: 100% (578/578), 58.92 MiB | 13.63 MiB/s, done.
Resolving deltas: 100% (70/70), done.
✅ SAM3 klasörü klonlandı
/content/sam3
Obtaining file:///content/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
... [48 lines truncated for the showcase repo] ...
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the pac


=== Kurulum Kontrolü ===
✅ SAM3 modülleri başarıyla import edildi!


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [1]:
import torch
print(f"GPU var mı? {torch.cuda.is_available()}")
print(f"GPU adı: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Yok'}")
print(f"GPU belleği: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU var mı? True
GPU adı: NVIDIA A100-SXM4-40GB
GPU belleği: 42.5 GB


In [2]:
CATEGORIES = [
    {"id": 1, "name": "bed", "supercategory": "furniture"},
    {"id": 2, "name": "cabinet", "supercategory": "furniture"},
    {"id": 3, "name": "carpet", "supercategory": "furniture"},
    {"id": 4, "name": "chair", "supercategory": "furniture"},
    {"id": 5, "name": "lamp", "supercategory": "furniture"},
    {"id": 6, "name": "shelf", "supercategory": "furniture"},
    {"id": 7, "name": "sofa", "supercategory": "furniture"},
    {"id": 8, "name": "table", "supercategory": "furniture"},
]

In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/furniture-retrieval")
# RAW_IMAGES_DIR = DRIVE_ROOT / "bedroom_images"
RAW_IMAGES_DIR = DRIVE_ROOT / "raw-images" / "living_room_images"
# PROCESSED_DIR = DRIVE_ROOT  / "bedroom_masks"
PROCESSED_DIR = DRIVE_ROOT  / "processed-images" / "living_room_masks"
MASK_DIR = PROCESSED_DIR

for folder in [RAW_IMAGES_DIR, PROCESSED_DIR, MASK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

len(list(RAW_IMAGES_DIR.glob("*.jpg")))

3605

In [4]:
import os
import json
import time
import torch
import numpy as np
from PIL import Image
from datetime import datetime
from pycocotools import mask as mask_utils
from tqdm import tqdm
#################################### For Image ####################################
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# GPU kontrolü ve ayarı
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    # GPU performansını artır
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Get all images from RAW_IMAGES_DIR
image_files = sorted(list(RAW_IMAGES_DIR.glob("*.jpg")))
if not image_files:
    raise FileNotFoundError(f"No images found in {RAW_IMAGES_DIR}")

print(f"\n{'='*60}")
print(f"📁 Total images to process: {len(image_files)}")
print(f"🏷️  Categories per image: {len(CATEGORIES)}")
print(f"📊 Total operations: {len(image_files)} × {len(CATEGORIES)} = {len(image_files) * len(CATEGORIES)}")
print(f"{'='*60}\n")

# Load the model once
print("Model yükleniyor...")
model_load_start = time.time()
model = build_sam3_image_model()
model = model.to(device)
model.eval()  # Inference modu
print(f"Model yükleme süresi: {time.time() - model_load_start:.2f} saniye\n")

processor = Sam3Processor(model)

total_start_time = time.time()
total_annotations = 0
skipped_count = 0

# Process each image
for img_idx, image_path in enumerate(tqdm(image_files, desc="📸 Processing images"), start=1):
    IMAGE_PATH = str(image_path)
    OUTPUT_JSON = str(PROCESSED_DIR / (image_path.stem + ".json"))

    # ✅ Kontrol: JSON dosyası daha önce oluşturulmuş mu?
    if os.path.exists(OUTPUT_JSON):
        # JSON dosyasını oku ve geçerli olup olmadığını kontrol et
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
                # Dosyada annotation varsa atla
                if existing_data.get('annotations'):
                    existing_ann_count = len(existing_data['annotations'])
                    total_annotations += existing_ann_count
                    skipped_count += 1
                    tqdm.write(f"⏭️  [{img_idx}/{len(image_files)}] {image_path.name} | Zaten işlenmiş (Ann:{existing_ann_count}) - Atlandı")
                    continue
        except (json.JSONDecodeError, KeyError):
            # JSON bozuksa yeniden işle
            tqdm.write(f"⚠️  [{img_idx}/{len(image_files)}] {image_path.name} | JSON bozuk - Yeniden işleniyor")

    image_start_time = time.time()

    # Load image
    image = Image.open(IMAGE_PATH).convert("RGB")

    # Initialize COCO format for this image
    coco_output = {
        "info": {
            "description": "SAM3 Generated Furniture Dataset",
            "version": "1.0",
            "year": datetime.now().year,
            "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        },
        "licenses": [],
        "images": [{
            "id": 1,
            "file_name": os.path.basename(IMAGE_PATH),
            "width": image.width,
            "height": image.height
        }],
        "annotations": [],
        "categories": CATEGORIES
    }

    annotation_id = 1
    image_id = 1
    image_annotations = 0

    # Process each category for this image
    for category in CATEGORIES:
        category_id = category["id"]
        category_name = category["name"]

        with torch.no_grad():
            inference_state = processor.set_image(image)
            output = processor.set_text_prompt(state=inference_state, prompt=category_name)

        # Get the masks, bounding boxes, and scores
        masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

        if masks is not None and len(masks) > 0:
            for idx, mask in enumerate(masks):
                # Convert mask to numpy
                mask_np = mask.detach().cpu().numpy() if torch.is_tensor(mask) else mask
                if mask_np.ndim > 2:
                    mask_np = mask_np.squeeze()
                mask_bin = (mask_np > 0).astype(np.uint8)

                if mask_bin.sum() == 0:
                    continue

                # Encode mask to RLE
                rle = mask_utils.encode(np.asfortranarray(mask_bin))
                rle["counts"] = rle["counts"].decode("utf-8")

                # Get bounding box from model output
                if boxes is not None and len(boxes) > idx:
                    box_np = boxes[idx].detach().cpu().numpy() if torch.is_tensor(boxes[idx]) else boxes[idx]
                    x1, y1, x2, y2 = box_np
                    bbox = [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]
                else:
                    bbox = mask_utils.toBbox(rle).tolist()

                area = float(mask_utils.area(rle))

                # Create annotation
                annotation = {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": category_id,
                    "segmentation": rle,
                    "bbox": bbox,
                    "area": area,
                    "iscrowd": 0
                }

                # Add score if available
                if scores is not None and len(scores) > idx:
                    score_val = scores[idx]
                    score_val = score_val.item() if torch.is_tensor(score_val) else score_val
                    annotation["score"] = float(score_val)

                coco_output["annotations"].append(annotation)
                annotation_id += 1
                image_annotations += 1

    # Save COCO JSON for this image
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, ensure_ascii=False, indent=2)

    image_time = time.time() - image_start_time
    total_annotations += image_annotations
    elapsed_time = time.time() - total_start_time
    remaining_images = len(image_files) - img_idx
    avg_time_per_image = elapsed_time / img_idx
    estimated_remaining_time = remaining_images * avg_time_per_image

    # Print progress for this image (compact format)
    category_summary = []
    if image_annotations > 0:
        for cat in CATEGORIES:
            count = sum(1 for ann in coco_output['annotations'] if ann['category_id'] == cat['id'])
            if count > 0:
                category_summary.append(f"{cat['name']}:{count}")

    categories_str = f"[{', '.join(category_summary)}]" if category_summary else "[hiç nesne yok]"

    tqdm.write(f"✅ [{img_idx}/{len(image_files)}] {image_path.name} | Ann:{image_annotations} | Süre:{image_time:.1f}s | Toplam:{total_annotations} | {categories_str}")
    tqdm.write(f"⏱️  Geçen:{elapsed_time/60:.1f}dk | Kalan:{remaining_images} resim (~{estimated_remaining_time/60:.1f}dk) | Ort:{avg_time_per_image:.1f}s/resim")

total_time = time.time() - total_start_time

print(f"\n{'='*60}")
print(f"✅ ALL IMAGES COMPLETED!")
print(f"{'='*60}")
print(f"📁 Total images processed: {len(image_files)}")
print(f"⏭️  Skipped (already processed): {skipped_count}")
print(f"🔄 Newly processed: {len(image_files) - skipped_count}")
print(f"📊 Total annotations: {total_annotations}")
print(f"📈 Average annotations per image: {total_annotations / len(image_files):.1f}")
print(f"⏱️  Total time: {total_time:.2f}s ({total_time/60:.1f} min)")
print(f"⚡ Average time per image: {total_time / len(image_files):.2f}s")
print(f"📂 Output directory: {PROCESSED_DIR}")
print(f"{'='*60}")

Kullanılan cihaz: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.56 GB

📁 Total images to process: 3605
🏷️  Categories per image: 8
📊 Total operations: 3605 × 8 = 28840

Model yükleniyor...
Model yükleme süresi: 22.62 saniye

✅ [1/3605] <image>.jpg | Ann:6 | Süre:11.8s | Toplam:6 | [carpet:1, sofa:3, table:2]
⏱️  Geçen:0.2dk | Kalan:3604 resim (~711.2dk) | Ort:11.8s/resim
✅ [2/3605] <image>.jpg | Ann:12 | Süre:2.2s | Toplam:18 | [cabinet:3, carpet:1, chair:1, shelf:6, sofa:1]
⏱️  Geçen:0.2dk | Kalan:3603 resim (~420.5dk) | Ort:7.0s/resim
✅ [3/3605] <image>.jpg | Ann:17 | Süre:2.2s | Toplam:35 | [carpet:3, chair:2, lamp:4, sofa:4, table:4]
⏱️  Geçen:0.3dk | Kalan:3602 resim (~324.3dk) | Ort:5.4s/resim
✅ [4/3605] <image>.jpg | Ann:11 | Süre:2.4s | Toplam:46 | [carpet:2, chair:2, lamp:2, sofa:1, table:4]
⏱️  Geçen:0.3dk | Kalan:3601 resim (~278.7dk) | Ort:4.6s/resim
✅ [5/3605] <image>.jpg | Ann:39 | Süre:2.2s | Toplam:85 | [cabinet:2, carpet:2, chair:3, shelf:31, table:1]
⏱️  Geçen:0.3dk 

  self.setter(val)
/it]📸 Processing images: 100%|█████████▉| 3602/3605 [2:17:00<00:06,  2.24s/it]📸 Processing images: 100%|█████████▉| 3603/3605 [2:17:02<00:04,  2.21s/it]📸 Processing images: 100%|█████████▉| 3604/3605 [2:17:04<00:02,  2.20s/it]📸 Processing images: 100%|██████████| 3605/3605 [2:17:07<00:00,  2.28s/it]